# Tutorial 1: Accessing EQ Synapse Data from Parquet Files

This tutorial shows how to load and plot power quality data directly from the parquet files
stored on the gateway. This is the most direct way to work with EQ Synapse data.

**What you will learn:**
1. Navigate the data directory structure
2. Load power monitor (PMon) and continuous point-on-wave (CPOW) data
3. Create plots using the `equser` library and custom matplotlib code

**Data overview:**
- **PMon files** contain aggregated power metrics (RMS voltage, current, power, frequency) recorded once per line cycle (~60 Hz). File names use the pattern `YYYYMMDD_HHMM.parquet`.
- **CPOW files** contain raw waveform samples at **32 kHz** (32,000 samples/second) for all voltage and current channels. File names use `YYYYMMDD_HHMMSS.parquet`. A single 60-second file contains ~1.92 million rows.

## 1. Import libraries

The `equser` package provides pre-built plotters, while `pyarrow` and `matplotlib` give you full control for custom analysis.

In [ ]:
# Standard libraries
from pathlib import Path

# Libraries that must be installed on your platform (e.g. using `pip install`)
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
from matplotlib.dates import DateFormatter

import equser as eq

%matplotlib inline

## 2. Create paths to the data

EQ Synapse stores data in `/var/lib/eq-watch/data/` with separate subdirectories for PMon and CPOW data.

In [ ]:
datadir = Path('/var/lib/eq-watch/data')  # Adjust if running outside the gateway
pmon_dir = datadir / 'pmon'  # Power monitor data directory
cpow_dir = datadir / 'cpow'  # Continuous point-on-wave (CPOW) data directory

## 3. List available files

### Power monitor files

PMon files are named `YYYYMMDD_HHMM.parquet` and typically cover one hour each.

In [ ]:
pmon_files = sorted(pmon_dir.glob('*.parquet'))
print(f"Found {len(pmon_files)} PMon files")
for fpath in pmon_files[-5:]:  # Show the 5 most recent
    print(f"  {fpath.name}")

In [ ]:
# Use the most recent file (or override with a specific filename)
pmon_selected_file = pmon_files[-1] if pmon_files else None
print(f"Selected: {pmon_selected_file}")

### CPOW waveform files

CPOW files are named `YYYYMMDD_HHMMSS.parquet` and typically cover 60 seconds each.
At 32 kHz sampling across 7 channels (VA, VB, VC, IA, IB, IC, IN), each file is quite large.

In [ ]:
cpow_files = sorted(cpow_dir.glob('*.parquet'))
print(f"Found {len(cpow_files)} CPOW files")
for fpath in cpow_files[-5:]:  # Show the 5 most recent
    print(f"  {fpath.name}")

In [ ]:
# Use the most recent file (or override with a specific filename)
cpow_selected_file = cpow_files[-1] if cpow_files else None
print(f"Selected: {cpow_selected_file}")

## 4. Create pre-established plots

The `equser` library includes built-in plotters that generate standard SVG plots with minimal code.

### Power monitor plots

In [ ]:
plotter = eq.plotting.PowerMonitorPlotter()
plotter.plot_file(pmon_selected_file)
# This will create plots as SVG images in your local directory.
# You can double-click an image file in the pane to the left to view it.

*Now double click to view a plot image file from the pane to the left.*

### CPOW waveform plots

In [ ]:
plotter = eq.plotting.WaveformPlotter()
plotter.plot_file(cpow_selected_file)
# This will create plots as SVG images in your local directory.
# You can double-click an image file in the pane to the left to view it.

*Now double click to view a plot image file from the pane to the left.*

## 5. Create custom plots

For full control over plotting, load the parquet data directly and use matplotlib.

### Power monitor plots

PMon columns include:
- **Voltage RMS**: AVRMS, BVRMS, CVRMS (and fundamental: AFVRMS, BFVRMS, CFVRMS)
- **Current RMS**: AIRMS, BIRMS, CIRMS, NIRMS (and fundamental: AFIRMS, BFIRMS, CFIRMS)
- **Power**: AWATT, BWATT, CWATT (and fundamental real/reactive: AFWATT, BFWATT, CFWATT, AFVAR, BFVAR, CFVAR)
- **Frequency**: FREQ

In [ ]:
# First, load the data and extract the time column.
table = pq.read_table(pmon_selected_file)
time = np.array(table['time_us'], dtype='datetime64[us]')
print("Available signals are:")
print(", ".join(name for name in table.column_names if name != 'time_us'))

In [ ]:
# Now create a plot from selected columns.
fig, ax = plt.subplots()
ax.set_title('RMS Voltage')
ax.plot(time, table['AVRMS'], 'k', label='AVRMS', zorder=3, alpha=0.7)
ax.plot(time, table['BVRMS'], 'r', label='BVRMS', zorder=2, alpha=0.7)
ax.plot(time, table['CVRMS'], 'b', label='CVRMS', zorder=1, alpha=0.7)
ax.set_ylabel("RMS voltage [V]")
ax.set_xlabel("Time (UTC)")
ax.xaxis.set_major_formatter(DateFormatter('%H:%M'))
ax.legend()
plt.show()
plt.close()

### CPOW waveform plots

CPOW columns are raw integer samples that must be scaled by `vscale` and `iscale` factors
stored in the parquet file metadata.

| Column | Description |
|--------|-------------|
| VA, VB, VC | Phase A, B, C voltage samples |
| IA, IB, IC | Phase A, B, C current samples |
| IN | Neutral current samples |

In [ ]:
# Load the CPOW data
table = pq.read_table(cpow_selected_file)
print("Available signals:")
print(", ".join(table.column_names))

In [ ]:
# Extract the voltage and current scaling factors from parquet metadata
parquet_file = pq.ParquetFile(cpow_selected_file)
file_metadata = parquet_file.metadata
user_metadata = file_metadata.metadata
vscale = float(user_metadata[b'vscale'].decode())
iscale = float(user_metadata[b'iscale'].decode())
print(f"Voltage scale: {vscale}, Current scale: {iscale}")

In [ ]:
# Take a slice of the waveform data to plot.
SAMPLE_RATE_HZ = 32_000
start_sec = 0
duration_ms = 100
start_sample = int(start_sec * SAMPLE_RATE_HZ)
num_samples = int(duration_ms * SAMPLE_RATE_HZ / 1000)
our_slice = slice(start_sample, start_sample + num_samples)
num_sliced_samples = len(table['VA'][our_slice])
time = np.arange(0, num_sliced_samples / SAMPLE_RATE_HZ * 1000, 1 / SAMPLE_RATE_HZ * 1000)

In [ ]:
# Finally, plot voltage and current waveforms.
fig, ax = plt.subplots()
ax.set_title("Voltage Waveforms")
ax.set_xlabel("Elapsed time [ms]")
ax.set_ylabel("Voltage [V]")
ax.plot(time, table['VA'][our_slice].to_numpy() * vscale, 'k', label='VA', zorder=3, alpha=0.7)
ax.plot(time, table['VB'][our_slice].to_numpy() * vscale, 'r', label='VB', zorder=3, alpha=0.7)
ax.plot(time, table['VC'][our_slice].to_numpy() * vscale, 'b', label='VC', zorder=3, alpha=0.7)
ax.legend(bbox_to_anchor=(1.0, 0.5))
plt.show()
plt.close()

fig, ax = plt.subplots()
ax.set_title("Current Waveforms")
ax.set_xlabel("Elapsed time [ms]")
ax.set_ylabel("Current [A]")
ax.plot(time, table['IA'][our_slice].to_numpy() * iscale, 'k', label='IA', zorder=3, alpha=0.7)
ax.plot(time, table['IB'][our_slice].to_numpy() * iscale, 'r', label='IB', zorder=3, alpha=0.7)
ax.plot(time, table['IC'][our_slice].to_numpy() * iscale, 'b', label='IC', zorder=3, alpha=0.7)
ax.legend(bbox_to_anchor=(1.0, 0.5))
plt.show()
plt.close()

## Next steps

- **Tutorial 2** (`02-local-duckdb.ipynb`): Query and aggregate these same parquet files with SQL using DuckDB.
- **Tutorial 3** (`03-backend-api.ipynb`): Query data through the gateway's REST API instead of reading files directly.
- **Harmonic analysis** (`../analysis/harmonic-analysis.ipynb`): Run FFT-based harmonic analysis on CPOW waveform data.
- **Power trends** (`../analysis/power-trends.ipynb`): Analyze long-term voltage, frequency, and power trends across multiple PMon files.